# Wavelet-YOLOv12 — Chen Split (Tuberculosis6208) — 5-Fold CV

Inline training notebook — `model.train()` dan semua hyperparam terlihat langsung di cell.

**Setup:**
- Dataset zip di Drive: `MyDrive/Tuberculosis6208.zip` (Pascal-VOC format)
- Chen test holdout (fixed): 101 images, `SPLIT_SEED=1050` (deterministic)
- 5-fold CV pada 1164 train+val: ≈931 train / ≈233 val per fold
- Logging: **W&B** — project `wavelet_yolo12_chen`, group per run name (5 fold runs + 1 summary run)

**Runtime:** A100 ≈ 25–30 menit per fold → ≈ 2.5 jam untuk full 5-fold sweep.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone repo (branch `dev/wavelet`)

In [2]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/wavelet-yolo12')
BRANCH   = 'dev/wavelet'

if REPO_DIR.exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull --ff-only
else:
    !git clone -b {BRANCH} https://github.com/iswantosan/wavelet-yolo12.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
!git log -1 --oneline

Cloning into '/content/wavelet-yolo12'...
remote: Enumerating objects: 1326, done.
remote: Counting objects: 100% (1326/1326), done.
remote: Compressing objects: 100% (717/717), done.
remote: Total 1326 (delta 633), reused 1274 (delta 581), pack-reused 0 (from 0)
Receiving objects: 100% (1326/1326), 1.99 MiB | 13.21 MiB/s, done.
Resolving deltas: 100% (633/633), done.
cwd: /content/wavelet-yolo12
9c4093e (HEAD -> dev/wavelet, origin/dev/wavelet) fix nwd ration param


## 3. Install dependencies (editable, supaya `WaveDown` ke-load)

In [3]:
!pip -q install -e . wandb

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done


In [4]:
import torch, ultralytics
from ultralytics.nn.modules import WaveDown, HaarDWT
print('torch       :', torch.__version__, '| cuda:', torch.cuda.is_available())
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('ultralytics :', ultralytics.__version__)
print('WaveDown OK :', WaveDown is not None and HaarDWT is not None)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
torch       : 2.11.0+cu128 | cuda: True
GPU         : NVIDIA A100-SXM4-40GB
ultralytics : 8.3.63
WaveDown OK : True


## 4. Build Chen split (1024 / 140 / 101, seed=42)

Extract zip → konversi VOC XML → YOLO `.txt` → deterministic shuffle → tulis `data.yaml`. Skip kalau output sudah ada.

Output ini dipakai untuk:
- **test holdout** (101 images, fixed di semua fold)
- **pool train+val** (1164 images) yang nanti dipecah jadi 5 fold

In [5]:
DRIVE_ZIP    = '/content/drive/MyDrive/Tuberculosis6208.zip'
EXTRACT_DIR  = '/content/dataset/raw'
RAW_DIR      = f'{EXTRACT_DIR}/tuberculosis-phonecamera'
SPLIT_DIR    = '/content/tb_chen_split'
CHEN_YAML    = f'{SPLIT_DIR}/data.yaml'

!python scripts/build_chen_split.py \
    --zip "{DRIVE_ZIP}" --extract-dir "{EXTRACT_DIR}" \
    --src "{RAW_DIR}" --out "{SPLIT_DIR}"

!ls -la {SPLIT_DIR} && echo '---' && cat {CHEN_YAML}

Extracting /content/drive/MyDrive/Tuberculosis6208.zip -> /content/dataset/raw
Image+XML pairs: 1265 (target 1265)
Split: train=1024  val=140  test=101  seed=42

Wrote /content/tb_chen_split/data.yaml
total 24
drwxr-xr-x 5 root root 4096 Jun  2 05:55 .
drwxr-xr-x 1 root root 4096 Jun  2 05:55 ..
-rw-r--r-- 1 root root  205 Jun  2 05:55 data.yaml
drwxr-xr-x 4 root root 4096 Jun  2 05:55 test
drwxr-xr-x 4 root root 4096 Jun  2 05:55 train
drwxr-xr-x 4 root root 4096 Jun  2 05:55 val
---
# Chen-style split (Chen et al. IJAI 2024) — 1024/140/101
# Split seed: 42 (deterministic)
path: /content/tb_chen_split
train: train/images
val:   val/images
test:  test/images
nc: 1
names:
  0: bacilli


## 5. Smoke test (build model + dummy forward)

In [6]:
!python scripts/smoke_test_wavelet.py

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.

=== ultralytics/cfg/models/v12/yolov12s.yaml (scale=n) ===
Overriding model.yaml nc=80 with nc=2
  params : 9.10 M
  output : [(1, 6, 8400)]

=== ultralytics/cfg/models/v12/yolov12s-wavelet-p3.yaml (scale=n) ===
Overriding model.yaml nc=80 with nc=2
  params : 9.16 M
  output : [(1, 6, 8400)]

=== ultralytics/cfg/models/v12/yolov12s-wavelet.yaml (scale=n) ===
Overriding model.yaml nc=80 with nc=2
  params : 8.83 M
  output : [(1, 6, 8400)]

OK


## 6. W&B login

Paste API key dari https://wandb.ai/authorize ketika di-prompt.

In [7]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: is-san86 (is-san86-binus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 7. Config

Ganti `MODEL_CFG` ke salah satu (filename ber-suffix `s` → scale `s` auto-detected → ~9.1M params, match `yolov12s.pt` pretrained):
- `ultralytics/cfg/models/v12/yolov12s.yaml` — baseline (no wavelet)
- `ultralytics/cfg/models/v12/yolov12s-wavelet-p3.yaml` — WaveDown di P3 saja
- `ultralytics/cfg/models/v12/yolov12s-wavelet.yaml` — WaveDown di P3+P4+P5 (default)

In [8]:
MODEL_CFG    = "ultralytics/cfg/models/v12/yolov12s.yaml"   # scale s -> 9.1M params
PRETRAINED   = "yolov12s.pt"       # auto-download, matches scale
SEED         = 1050
EPOCHS       = 60
IMGSZ        = 640
BATCH        = 16
DEVICE       = 0

# K-fold settings
N_FOLDS      = 5
KFOLD_SEED   = 1050      # deterministic fold assignment
KFOLD_DIR    = '/content/tb_kfold'

WANDB_PROJECT = "wavelet_yolo12_chen"
RUN_PROJECT   = "/content/runs/wavelet_chen"
RUN_BASE      = f"{Path(MODEL_CFG).stem}_seed{SEED}_{EPOCHS}ep_kf{N_FOLDS}"
GROUP_NAME    = RUN_BASE   # all fold runs share this group in W&B

print("cfg     :", MODEL_CFG)
print("seed    :", SEED)
print("epochs  :", EPOCHS)
print("n_folds :", N_FOLDS)
print("group   :", GROUP_NAME)

cfg     : ultralytics/cfg/models/v12/yolov12s.yaml
seed    : 1050
epochs  : 60
n_folds : 5
group   : yolov12s_seed1050_60ep_kf5


## 8. Build 5-fold splits (inline)

Pool 1164 images (Chen train + Chen val), deterministic shuffle dengan `KFOLD_SEED=42`, pecah jadi 5 fold. Tiap fold:
- `train/`: 4 fold lain (≈931 imgs)
- `val/`:   1 fold (≈233 imgs)
- `test/`:  Chen holdout (101 imgs, sama di semua fold)

Pakai symlink supaya cepat dan hemat disk.

In [9]:
import random, shutil
from pathlib import Path

chen = Path(SPLIT_DIR)
kfold = Path(KFOLD_DIR)

IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def list_imgs(d: Path):
    return sorted([p for p in d.glob('*') if p.suffix.lower() in IMG_EXTS])

def label_for(img: Path) -> Path:
    return img.parent.parent / 'labels' / (img.stem + '.txt')

def sym(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src.resolve())

train_imgs = list_imgs(chen / 'train' / 'images')
val_imgs   = list_imgs(chen / 'val' / 'images')
test_imgs  = list_imgs(chen / 'test' / 'images')
pool = train_imgs + val_imgs
print(f'Pool train+val : {len(pool)} images')
print(f'Test holdout   : {len(test_imgs)} images (fixed)')

assert len(pool) > 0, (
    f'Empty pool — pastikan section 4 (build Chen split) sudah jalan dan menghasilkan images di '
    f'{chen}/train/images dan {chen}/val/images'
)
assert len(test_imgs) > 0, f'Empty test set — periksa {chen}/test/images'

rng = random.Random(KFOLD_SEED)
shuffled = list(pool)
rng.shuffle(shuffled)

fold_size = len(shuffled) // N_FOLDS
folds = [shuffled[i*fold_size:(i+1)*fold_size] for i in range(N_FOLDS)]
# Distribute remainder to earliest folds
for i, img in enumerate(shuffled[N_FOLDS*fold_size:]):
    folds[i].append(img)

# Fresh build
if kfold.exists():
    shutil.rmtree(kfold)

FOLD_YAMLS = []
for k in range(N_FOLDS):
    val_k   = folds[k]
    val_set = set(val_k)
    train_k = [img for img in shuffled if img not in val_set]

    fold_dir = kfold / f'fold{k}'
    fold_dir.mkdir(parents=True, exist_ok=True)   # ensure dir exists even if all groups empty

    for split_name, group in (('train', train_k), ('val', val_k), ('test', test_imgs)):
        for img in group:
            sym(img, fold_dir / split_name / 'images' / img.name)
            lbl = label_for(img)
            if lbl.exists():
                sym(lbl, fold_dir / split_name / 'labels' / (img.stem + '.txt'))

    yml = fold_dir / 'data.yaml'
    yml.write_text(
        f'# 5-fold CV — fold {k}/{N_FOLDS-1} (kfold_seed={KFOLD_SEED})\n'
        f'# train/val from Chen 1164-image pool; test = Chen 101-image holdout (fixed)\n'
        f'path: {fold_dir.resolve()}\n'
        'train: train/images\n'
        'val:   val/images\n'
        'test:  test/images\n'
        'nc: 1\n'
        'names:\n'
        '  0: bacilli\n'
    )
    FOLD_YAMLS.append(str(yml))
    print(f'  fold{k}: train={len(train_k):4d}  val={len(val_k):3d}  test={len(test_imgs):3d}  ->  {yml}')

print(f'\nAll {N_FOLDS} fold yamls ready under {kfold}')

Pool train+val : 1164 images
Test holdout   : 101 images (fixed)
  fold0: train= 931  val=233  test=101  ->  /content/tb_kfold/fold0/data.yaml
  fold1: train= 931  val=233  test=101  ->  /content/tb_kfold/fold1/data.yaml
  fold2: train= 931  val=233  test=101  ->  /content/tb_kfold/fold2/data.yaml
  fold3: train= 931  val=233  test=101  ->  /content/tb_kfold/fold3/data.yaml
  fold4: train= 932  val=232  test=101  ->  /content/tb_kfold/fold4/data.yaml

All 5 fold yamls ready under /content/tb_kfold


## 9. Seed + Ultralytics callback setup

Disable built-in W&B callback — kita log manual per fold.

In [10]:
import os, gc, random, numpy as np, torch

# Stable SDP kernel (avoid flash/mem-efficient mismatch on Ampere/Ada)
os.environ['PYTORCH_SDP_KERNEL'] = 'math'
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Disable Ultralytics' built-in W&B callback — kita log manual
from ultralytics.utils import SETTINGS
SETTINGS.update({'wandb': False})
print('Seed + SDP kernel + Ultralytics W&B callback disabled.')

Seed + SDP kernel + Ultralytics W&B callback disabled.


## 10. Helper functions (eval + W&B csv-replay)

In [11]:
import pandas as pd

EVAL_KEYS = ('mAP50', 'mAP50-95', 'mAP@0.9', 'precision', 'recall')

def evaluate(model, data_yaml, split):
    """Run model.val() on the given split and return metrics dict."""
    eva = model.val(data=data_yaml, split=split, imgsz=IMGSZ, device=DEVICE, verbose=False)
    out = {
        'mAP50':     float(eva.box.map50),
        'mAP50-95':  float(eva.box.map),
        'precision': float(np.mean(np.atleast_1d(eva.box.p))),
        'recall':    float(np.mean(np.atleast_1d(eva.box.r))),
        'mAP@0.9':   float('nan'),
    }
    try:
        ap_all = eva.box.all_ap
        if ap_all is not None and len(ap_all):
            ap = ap_all.mean(axis=0) if (hasattr(ap_all, 'ndim') and ap_all.ndim == 2) else ap_all
            if len(ap) >= 9:
                out['mAP@0.9'] = float(ap[8])
    except Exception as e:
        print(f'  (mAP@0.9 extract failed: {e})')
    return out


def log_csv_to_wandb(run, csv_path):
    """Replay results.csv epoch-by-epoch into the active W&B run."""
    wandb.define_metric('epoch')
    for k in [
        'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'train/total_loss',
        'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'val/total_loss',
        'val/mAP50', 'val/mAP50-95', 'val/precision', 'val/recall', 'lr/pg0',
    ]:
        wandb.define_metric(k, step_metric='epoch')

    if not Path(csv_path).exists():
        print(f'  results.csv missing: {csv_path}')
        return
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    col_map = [
        ('train/box_loss', 'train/box_loss'),
        ('train/cls_loss', 'train/cls_loss'),
        ('train/dfl_loss', 'train/dfl_loss'),
        ('val/box_loss', 'val/box_loss'),
        ('val/cls_loss', 'val/cls_loss'),
        ('val/dfl_loss', 'val/dfl_loss'),
        ('metrics/mAP50(B)', 'val/mAP50'),
        ('metrics/mAP50-95(B)', 'val/mAP50-95'),
        ('metrics/precision(B)', 'val/precision'),
        ('metrics/recall(B)', 'val/recall'),
        ('lr/pg0', 'lr/pg0'),
    ]
    for _, row in df.iterrows():
        try: ep = int(row.get('epoch', 0))
        except Exception: continue
        log = {'epoch': ep}
        for src, dst in col_map:
            if src in df.columns:
                try: log[dst] = float(row[src])
                except Exception: pass
        tb, tc, td = log.get('train/box_loss'), log.get('train/cls_loss'), log.get('train/dfl_loss')
        if None not in (tb, tc, td): log['train/total_loss'] = tb + tc + td
        vb, vc, vd = log.get('val/box_loss'), log.get('val/cls_loss'), log.get('val/dfl_loss')
        if None not in (vb, vc, vd): log['val/total_loss'] = vb + vc + vd
        run.log(log)
    print(f'  Logged {len(df)} epoch rows to W&B.')


def upload_plots(run, save_dir):
    for img in Path(save_dir).glob('*.png'):
        if any(t in img.stem.lower() for t in ('results', 'confusion', 'f1_curve', 'pr_curve', 'p_curve', 'r_curve')):
            try: run.log({f'plots/{img.stem}': wandb.Image(str(img))})
            except Exception: pass

print('Helpers ready.')

Helpers ready.


## 11. K-fold training loop

Tiap fold = satu W&B run dengan `group=GROUP_NAME` (semua run sharing group). Per fold dilakukan:
1. Train (`EPOCHS` epoch) dengan `data.yaml` fold tersebut
2. Log per-epoch curves dari `results.csv`
3. Evaluasi `best.pt` di **val** (fold-specific) dan **test** (Chen holdout)
4. Log summary metrics ke W&B, cleanup GPU/RAM

Total ≈ `N_FOLDS × EPOCHS` epoch — siapkan koneksi Colab yang stabil.

In [12]:
import time
from ultralytics import YOLO

all_results = []

for k, fold_yaml in enumerate(FOLD_YAMLS):
    run_name = f'{RUN_BASE}_fold{k}'
    print(f'\n{"="*70}\n  FOLD {k}/{N_FOLDS-1}  ->  {run_name}\n{"="*70}')

    run = wandb.init(
        project=WANDB_PROJECT,
        group=GROUP_NAME,
        name=run_name,
        reinit=True,
        job_type='train',
        config=dict(
            fold=k, n_folds=N_FOLDS, kfold_seed=KFOLD_SEED,
            model_cfg=MODEL_CFG, data_yaml=fold_yaml, pretrained=PRETRAINED,
            seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            optimizer='SGD', lr0=0.01, momentum=0.937, cos_lr=True,
            split=f'kfold{N_FOLDS}_chen_holdout',
        ),
        tags=[Path(MODEL_CFG).stem, f'seed{SEED}', f'kfold{N_FOLDS}', f'fold{k}'],
    )
    print('  W&B run:', run.url)

    # ---- Train ----
    model = YOLO(MODEL_CFG)
    try:
        model.load(PRETRAINED)
        print(f'  Loaded pretrained: {PRETRAINED}')
    except Exception as e:
        print(f'  [warn] could not load pretrained: {e}')

    t0 = time.time()
    results = model.train(
        data=fold_yaml,
        nwd_ratio=0.5,
        nwd_c=12.8,
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        optimizer='SGD', lr0=0.01, momentum=0.937, cos_lr=True, patience=0,
        amp=True, deterministic=True, seed=SEED, workers=8,
        hsv_h=0.1, hsv_s=0.3, hsv_v=0.3,
        degrees=10, translate=0.05, scale=0.3, shear=0.0, perspective=0.0,
        flipud=0.5, fliplr=0.5,
        mosaic=0.3, mixup=0.3, auto_augment=None,
        project=RUN_PROJECT,
        name=run_name,
        exist_ok=True, save=True, verbose=True,
    )
    train_secs = time.time() - t0
    print(f'  Train time: {train_secs/60:.1f} min   Save dir: {results.save_dir}')

    # ---- Replay per-epoch curves to W&B ----
    log_csv_to_wandb(run, Path(results.save_dir) / 'results.csv')

    # ---- Eval best.pt on val (fold-specific) and test (Chen holdout) ----
    best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
    print(f'  Best ckpt: {best_pt}')
    eval_model = YOLO(str(best_pt))
    val_metrics  = evaluate(eval_model, fold_yaml, 'val')
    test_metrics = evaluate(eval_model, fold_yaml, 'test')

    print(f'\n  === FOLD {k} RESULTS ===')
    print(f'  VAL : ' + '  '.join(f'{m}={val_metrics[m]:.4f}'  for m in EVAL_KEYS))
    print(f'  TEST: ' + '  '.join(f'{m}={test_metrics[m]:.4f}' for m in EVAL_KEYS))

    # ---- Summary metrics to W&B ----
    for m, v in val_metrics.items():  run.summary[f'val/{m}']  = v
    for m, v in test_metrics.items(): run.summary[f'test/{m}'] = v
    run.summary['train/time_min'] = train_secs / 60

    upload_plots(run, results.save_dir)
    run.finish()

    all_results.append({
        'fold': k,
        'val':  val_metrics,
        'test': test_metrics,
        'train_min': train_secs / 60,
        'save_dir': str(results.save_dir),
    })

    # ---- Cleanup before next fold ----
    del model, eval_model, results
    torch.cuda.empty_cache(); gc.collect()

print(f'\n{"="*70}\nDone — {N_FOLDS} folds finished.\n{"="*70}')


  FOLD 0/4  ->  yolov12s_seed1050_60ep_kf5_fold0


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  W&B run: https://wandb.ai/is-san86-binus/wavelet_yolo12_chen/runs/z9hudkyl


100%|██████████| 17.8M/17.8M [00:00<00:00, 113MB/s]


Transferred 739/739 items from pretrained weights
  Loaded pretrained: yolov12s.pt
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=ultralytics/cfg/models/v12/yolov12s.yaml, data=/content/tb_kfold/fold0/data.yaml, epochs=60, time=None, patience=0, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=/content/runs/wavelet_chen, name=yolov12s_seed1050_60ep_kf5_fold0, exist_ok=True, pretrained=yolov12s.pt, optimizer=SGD, verbose=True, seed=1050, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=Fals

...

val: New cache created: /content/tb_kfold/fold4/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:02<00:00,  2.97it/s]


                   all        101        898      0.808      0.792      0.873      0.434
Speed: 0.1ms preprocess, 2.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/wavelet-yolo12/runs/detect/val10

  === FOLD 4 RESULTS ===
  VAL : mAP50=0.8635  mAP50-95=0.4264  mAP@0.9=0.0044  precision=0.7827  recall=0.8009
  TEST: mAP50=0.8731  mAP50-95=0.4339  mAP@0.9=0.0056  precision=0.8079  recall=0.7917


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr/pg0,▃▆██████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/box_loss,█▄▅▅▄▄▄▄▃▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/cls_loss,█▅▅▅▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/dfl_loss,█▄▅▅▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
train/total_loss,█▇▆▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
val/box_loss,▅█▅▄▇▃▄▂▇▆▄▃▃▂▂▂▃▄▂▂▂▃▂▄▂▃▁▁▁▁▁▁▂▂▂▂▂▂▂▂
val/cls_loss,▂▂█▆▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/dfl_loss,▃█▅▆▃▃▃▂▃▆▄▂▃▃▃▂▂▃▁▃▂▂▂▂▂▃▂▂▁▁▁▁▁▂▁▁▂▁▁▁
val/mAP50,▂▁▂▁▃▅▅▃▅▆▆▅▆▆▇▇▆▇▇▅▇▇▇▇▇▇▇▇▇███████████
+4,...



Done — 5 folds finished.


## 12. Cross-fold aggregation (mean ± std)

Log a single summary run `<RUN_BASE>_SUMMARY` ke W&B yang berisi mean/std semua metrik val & test.

In [16]:
import math, statistics

def _valid(xs):
    return [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]

agg = {}
print(f'\n=== {N_FOLDS}-FOLD CV SUMMARY ({RUN_BASE}) ===\n')
print(f"{'Split/Metric':<22}{'Mean':>10}{'Std':>10}{'Min':>10}{'Max':>10}")
print('-' * 62)
for split in ('val', 'test'):
    for m in EVAL_KEYS:
        vals = _valid([f[split].get(m) for f in all_results])
        if not vals:
            continue
        mean = statistics.mean(vals)
        std  = statistics.stdev(vals) if len(vals) > 1 else 0.0
        agg[f'{split}/{m}/mean'] = mean
        agg[f'{split}/{m}/std']  = std
        agg[f'{split}/{m}/min']  = min(vals)
        agg[f'{split}/{m}/max']  = max(vals)
        print(f'{split}/{m:<16}{mean:>10.4f}{std:>10.4f}{min(vals):>10.4f}{max(vals):>10.4f}')

train_mins = _valid([f['train_min'] for f in all_results])
agg['train/time_min/mean'] = statistics.mean(train_mins) if train_mins else 0.0
agg['train/time_min/sum']  = sum(train_mins) if train_mins else 0.0
print('-' * 62)
print(f"train_min (avg/total)  {agg['train/time_min/mean']:>10.1f}{'':>10}{'':>10}{agg['train/time_min/sum']:>10.1f}")

# Log summary run
summary_run = wandb.init(
    project=WANDB_PROJECT,
    group=GROUP_NAME,
    name=f'{RUN_BASE}_SUMMARY',
    reinit=True,
    job_type='summary',
    config=dict(
        model_cfg=MODEL_CFG, n_folds=N_FOLDS, kfold_seed=KFOLD_SEED,
        seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    ),
    tags=[Path(MODEL_CFG).stem, f'kfold{N_FOLDS}', 'summary'],
)
for k, v in agg.items():
    summary_run.summary[k] = v
summary_run.summary['n_folds'] = N_FOLDS
# Also log a flat per-fold table
table = wandb.Table(columns=['fold'] + [f'val/{m}' for m in EVAL_KEYS] + [f'test/{m}' for m in EVAL_KEYS] + ['train_min'])
for f in all_results:
    table.add_data(
        f['fold'],
        *[f['val'].get(m, float('nan'))  for m in EVAL_KEYS],
        *[f['test'].get(m, float('nan')) for m in EVAL_KEYS],
        f['train_min'],
    )
summary_run.log({'per_fold_results': table})
summary_run.finish()
print(f'\nSummary run logged: {summary_run.name}')


=== 5-FOLD CV SUMMARY (yolov12s_seed1050_60ep_kf5) ===

Split/Metric                Mean       Std       Min       Max
--------------------------------------------------------------
val/mAP50               0.8681    0.0089    0.8590    0.8794
val/mAP50-95            0.4364    0.0097    0.4264    0.4468
val/mAP@0.9             0.0073    0.0025    0.0044    0.0095
val/precision           0.7870    0.0171    0.7670    0.8135
val/recall              0.8030    0.0211    0.7834    0.8369
test/mAP50               0.8729    0.0021    0.8703    0.8760
test/mAP50-95            0.4372    0.0071    0.4293    0.4476
test/mAP@0.9             0.0073    0.0015    0.0056    0.0092
test/precision           0.8115    0.0108    0.7958    0.8229
test/recall              0.7842    0.0123    0.7684    0.7996
--------------------------------------------------------------
train_min (avg/total)        12.9                          64.7


n_folds,5
test/mAP50-95/max,0.44763
test/mAP50-95/mean,0.43723
test/mAP50-95/min,0.42929
test/mAP50-95/std,0.00707
test/mAP50/max,0.876
test/mAP50/mean,0.87287
test/mAP50/min,0.87031
test/mAP50/std,0.00209
test/mAP@0.9/max,0.00924
+33,...



Summary run logged: yolov12s_seed1050_60ep_kf5_SUMMARY


## 13. (Opsional) Quick predict sample dari fold-0 best.pt

In [14]:
from ultralytics import YOLO

if all_results:
    best_pt = Path(all_results[0]['save_dir']) / 'weights' / 'best.pt'
    test_dir = Path(KFOLD_DIR) / 'fold0' / 'test' / 'images'
    pred_model = YOLO(str(best_pt))
    preds = pred_model.predict(
        source=str(test_dir),
        save=True, imgsz=IMGSZ, conf=0.25, device=DEVICE,
    )
    print('Predictions saved to:', preds[0].save_dir if preds else None)
else:
    print('No fold results to predict from.')


image 1/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0014.jpg: 480x640 12 bacillis, 93.6ms
image 2/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0052.jpg: 480x640 2 bacillis, 19.1ms
image 3/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0055.jpg: 480x640 20 bacillis, 18.0ms
image 4/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0062.jpg: 480x640 20 bacillis, 17.4ms
image 5/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0066.jpg: 480x640 17 bacillis, 17.5ms
image 6/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0089.jpg: 480x640 31 bacillis, 18.2ms
image 7/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0094.jpg: 480x640 11 bacillis, 17.3ms
image 8/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0115.jpg: 480x640 13 bacillis, 17.4ms
image 9/101 /content/tb_kfold/fold0/test/images/tuberculosis-phone-0136.jpg: 480x640 11 bacillis, 17.4ms
image 10/101 /content/tb_kfold/fold0/test/images/tuberc

In [15]:
!zip -r /content/runs.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/wavelet_chen/ (stored 0%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/ (stored 0%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/val_batch2_labels.jpg (deflated 8%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/train_batch0.jpg (deflated 10%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/val_batch2_pred.jpg (deflated 7%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/events.out.tfevents.1780382157.b3c3a0b5b9d5.1064.3 (deflated 92%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/confusion_matrix.png (deflated 37%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/train_batch1.jpg (deflated 6%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/R_curve.png (deflated 17%)
  adding: content/runs/wavelet_chen/yolov12s_seed1050_60ep_kf5_fold3/train_batch2950.jpg